# Z-Image (Turbo / Base) GGUF — stable-diffusion.cpp / Colab T4

Experimental notebook for **Z-Image-Turbo** by default (switchable to **Z-Image-Base** in cell 4) with GGUF. Uses **prebuilt CUDA sd-cli from Google Drive**, **no cloning/compilation required**. Default configuration: Turbo Q6 (~5.26 GB) + Qwen3-4B Q4 (~2.5 GB) + FLUX VAE (~335 MB).

**Execution order:** 1 → 2 → 3 → 4 → 5. `Turbo` uses **8 steps, CFG 1.0**; `Base` uses **30 steps, CFG 5.0** (steps are configurable). Do not use the 6-step schedule from MiniMax H3. Choose **Turbo** for maximum speed on T4 GPU, or **Base** to experiment with non-distilled baseline weights.

Sources: [Z-Image-Turbo GGUF](https://huggingface.co/leejet/Z-Image-Turbo-GGUF), [Z-Image-Base GGUF](https://huggingface.co/leejet/Z-Image-GGUF), [sd.cpp documentation](https://github.com/leejet/stable-diffusion.cpp/blob/master/docs/models/z_image.md).

Default resolution: 1920×1088. For fair comparison with H3, use identical long prompts, fixed seed, and increase to 2720×1536 once initial test succeeds. Z-Image is licensed under Apache-2.0.

In [ ]:
MODEL_SLUG = "z_image"

# @title 1. Runtime Setup / Drive Paths { display-mode: "form" }
import os
import sys
import json
import shutil
import pathlib
import subprocess
import time
from datetime import datetime

ROOT = pathlib.Path("/content")
DRIVE_BINARY_PATH = "/content/drive/MyDrive/sd_cpp_cuda_t4.tar.gz"  # @param {type:"string"}
MAX_VRAM_GB = 12  # @param {type:"slider", min:6, max:14, step:1}
SAVE_IMAGES_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/sdcpp_comparisons"  # @param {type:"string"}

MODELS_DIR = ROOT / "sdcpp_models" / MODEL_SLUG
OUTPUT_DIR = ROOT / "sdcpp_outputs" / MODEL_SLUG
BUILD_DIR = ROOT / "stable-diffusion.cpp" / "build"
SD_BIN = BUILD_DIR / "bin" / "sd-cli"
for path in (MODELS_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

try:
    import torch
    assert torch.cuda.is_available(), "Go to Runtime → Change runtime type → T4 GPU."
    gpu_name = torch.cuda.get_device_name(0)
    vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    ram_gib = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / (1024 ** 3)
    free_gib = shutil.disk_usage(ROOT).free / (1024 ** 3)
    print(f"GPU: {gpu_name} | VRAM: {vram_gib:.1f} GiB")
    print(f"Host RAM: {ram_gib:.1f} GiB | Free Disk: {free_gib:.1f} GiB")
    assert MAX_VRAM_GB < vram_gib, "MAX_VRAM_GB should be lower than actual GPU VRAM."
except ImportError as e:
    raise RuntimeError("Requires a Colab GPU runtime with PyTorch pre-installed.") from e

## 2. Restore `sd-cli` from Google Drive

Uses the archive created in the H3 notebook (`tar -czf ... -C /content/stable-diffusion.cpp/build bin`). Supports archives, direct binaries, or binary directories. No cloning or compilation steps.

In [ ]:
# @title 2. Restore CUDA Binary from Google Drive — NO BUILD { display-mode: "form" }
from google.colab import drive
import tarfile

drive.mount("/content/drive")
source = pathlib.Path(DRIVE_BINARY_PATH)
if not source.exists():
    raise FileNotFoundError(
        f"Not found: {source}\n"
        "Please adjust DRIVE_BINARY_PATH in cell 1. Default is the backup tar "
        "from MiniMax H3 notebook: sd_cpp_cuda_t4.tar.gz"
    )

BIN_DIR = BUILD_DIR / "bin"
BIN_DIR.mkdir(parents=True, exist_ok=True)
if source.is_file() and (source.name.endswith(".tar.gz") or source.suffix in (".tgz", ".tar")):
    # Legacy backup: tar -czf ... -C /content/stable-diffusion.cpp/build bin
    with tarfile.open(source, "r:*") as archive:
        for member in archive.getmembers():
            member_path = pathlib.PurePosixPath(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"Archive contains unsafe path: {member.name}")
        archive.extractall(BUILD_DIR, filter="data")
elif source.is_dir():
    # Drive directory may contain sd-cli directly or within a bin/ directory.
    dir_source = source / "bin" if (source / "bin" / "sd-cli").exists() else source
    if not (dir_source / "sd-cli").exists():
        raise FileNotFoundError(f"sd-cli not found in {dir_source}")
    shutil.copytree(dir_source, BIN_DIR, dirs_exist_ok=True)
elif source.is_file() and source.name in ("sd-cli", "sd"):
    shutil.copy2(source, SD_BIN)
else:
    raise ValueError("Supported formats: .tar.gz/.tgz/.tar, sd-cli file, or directory containing sd-cli.")

if not SD_BIN.is_file():
    raise FileNotFoundError(
        f"{SD_BIN} not found after restore. "
        "Please verify archive structure; expected bin/sd-cli."
    )
SD_BIN.chmod(SD_BIN.stat().st_mode | 0o111)
EXEC_ENV = os.environ.copy()
EXEC_ENV["LD_LIBRARY_PATH"] = f"{BIN_DIR}:{EXEC_ENV.get('LD_LIBRARY_PATH', '')}"
check = subprocess.run(
    [str(SD_BIN), "--version"], env=EXEC_ENV,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(check.stdout.strip()[:3000])
if check.returncode != 0:
    raise RuntimeError(
        "Binary cannot run in current Colab runtime. "
        "If .so libraries are missing, backup the entire build/bin folder rather than only sd-cli. "
        "If the binary is outdated, compile a new version once and save it to Drive."
    )
print("✅ Using restored binary:", SD_BIN)

## 3. Install Lightweight Python Dependencies

Installs only huggingface_hub for model downloading, Pillow for display, and nvidia-ml-py for VRAM measurement.

In [ ]:
# @title 3. Install Lightweight Dependencies — NO COMFYUI / DIFFUSERS
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2", "Pillow", "nvidia-ml-py"],
    check=True,
)
from huggingface_hub import hf_hub_download


def get_model(label, repo, filename):
    print(f"📥 {label}: {repo}/{filename}", flush=True)
    path = pathlib.Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODELS_DIR)))
    print(f"✅ {path.name} ({path.stat().st_size / 1e9:.2f} GB)", flush=True)
    return path

## 4. Download Weights by Variant

Choose `Turbo` or `Base` and download the 3 required weight files. For Base, Q4 and Q8 quants are available from leejet's GGUF repository.

In [ ]:
# @title 4. Select Z-Image Turbo / Base and Quantization { display-mode: "form" }
Z_VARIANT = "Turbo"  # @param ["Turbo", "Base"]
TURBO_QUANT = "Q6"  # @param ["Q4", "Q5", "Q6", "Q8"]
BASE_QUANT = "Q4"  # @param ["Q4", "Q8"]
Z_ENCODER = "Q4_K_M"  # @param ["Q4_K_M", "Q5_K_M", "Q6_K"]
STEPS_OVERRIDE = 0  # @param {type:"integer"}

TURBO_FILES = {
    "Q4": "z_image_turbo-Q4_K.gguf",
    "Q5": "z_image_turbo-Q5_0.gguf",
    "Q6": "z_image_turbo-Q6_K.gguf",
    "Q8": "z_image_turbo-Q8_0.gguf",
}
BASE_FILES = {
    "Q4": "z_image-Q4_0.gguf",
    "Q8": "z_image-Q8_0.gguf",
}
ENCODER_OPTIONS = {
    "Q4_K_M": "Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
    "Q5_K_M": "Qwen3-4B-Instruct-2507-Q5_K_M.gguf",
    "Q6_K": "Qwen3-4B-Instruct-2507-Q6_K.gguf",
}
is_turbo = Z_VARIANT == "Turbo"
model_repo = "leejet/Z-Image-Turbo-GGUF" if is_turbo else "leejet/Z-Image-GGUF"
model_file = TURBO_FILES[TURBO_QUANT] if is_turbo else BASE_FILES[BASE_QUANT]
MODEL_NAME = "Z-Image-Turbo" if is_turbo else "Z-Image-Base"
MODEL_VARIANT_NAME = Z_VARIANT
QUANT_LABEL = TURBO_QUANT if is_turbo else BASE_QUANT
CFG_SCALE = 1.0 if is_turbo else 5.0
STEPS = STEPS_OVERRIDE if STEPS_OVERRIDE > 0 else (8 if is_turbo else 30)

MODEL_PATHS = {
    "dit": get_model("DiT", model_repo, model_file),
    "encoder": get_model("Qwen3-4B", "unsloth/Qwen3-4B-Instruct-2507-GGUF", ENCODER_OPTIONS[Z_ENCODER]),
    "vae": get_model("VAE FLUX", "Comfy-Org/Z-Image-Turbo", "split_files/vae/ae.safetensors"),
}
print("Total weights size:", round(sum(p.stat().st_size for p in MODEL_PATHS.values()) / 1e9, 2), "GB")
print(f"Sampling schedule: {STEPS} steps, CFG {CFG_SCALE} ({Z_VARIANT})")

## 5. Image Generation and Benchmark

The notebook saves PNG, log, and JSON metadata recording execution time and VRAM usage. Compare Turbo with Turbo and Base with Base under identical conditions.

In [ ]:
from PIL import Image
from IPython.display import display
import threading

# @title 5. Generate Image + Save Benchmark { display-mode: "form" }
PROMPT = 'A photorealistic professional fashion photograph of an adult East Asian woman in her mid-twenties sitting gracefully on the stone edge of a luxurious outdoor swimming pool on a sunny summer afternoon. She has long flowing dark brown hair, natural makeup, realistic skin texture with subtle pores, anatomically natural brown eyes with aligned pupils and realistic catchlights, and a gentle smile. She wears an elegant floral-patterned one-piece swimsuit. Her hands rest naturally near her knees, and she looks directly at the camera. Behind her are turquoise water, tropical trees, neatly trimmed hedges, white lounge chairs, and a modern resort building. Medium portrait framing, her face is large and in sharp focus, eye-level 50mm lens, balanced natural daylight, authentic editorial photography, fine hair strands, believable anatomy, and realistic photographic detail.'  # @param {type:"string"}
WIDTH = 1920  # @param {type:"integer"}
HEIGHT = 1088  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
USE_FLASH_ATTENTION = True  # @param {type:"boolean"}
USE_VAE_TILING = True  # @param {type:"boolean"}

assert WIDTH >= 512 and HEIGHT >= 512, "Resolution should be at least 512px."
assert WIDTH % 32 == 0 and HEIGHT % 32 == 0, "Width and height must be divisible by 32."
assert 1 <= STEPS <= 100, "STEPS must be between 1 and 100."

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_id = f"{MODEL_SLUG}_{QUANT_LABEL}_{WIDTH}x{HEIGHT}_s{STEPS}_seed{SEED}_{stamp}"
image_path = OUTPUT_DIR / f"{run_id}.png"
log_path = OUTPUT_DIR / f"{run_id}.log"
metadata_path = OUTPUT_DIR / f"{run_id}.json"

cmd = [
    str(SD_BIN),
    "--diffusion-model", str(MODEL_PATHS["dit"]),
    "--llm", str(MODEL_PATHS["encoder"]),
    "--vae", str(MODEL_PATHS["vae"]),
    "-p", PROMPT,
    "-W", str(WIDTH), "-H", str(HEIGHT),
    "--seed", str(SEED), "--rng", "cpu",
    "--steps", str(STEPS), "--cfg-scale", str(CFG_SCALE),
    "--sampling-method", "euler",
    "--auto-fit", "on", "--max-vram", str(MAX_VRAM_GB),
    "-o", str(image_path),
]
if USE_FLASH_ATTENTION:
    cmd.append("--diffusion-fa")
if USE_VAE_TILING:
    cmd.append("--vae-tiling")

gpu_samples = []
monitor_stop = threading.Event()


def monitor_vram():
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        while not monitor_stop.is_set():
            gpu_samples.append(pynvml.nvmlDeviceGetMemoryInfo(handle).used / (1024 * 1024))
            monitor_stop.wait(1.0)
        pynvml.nvmlShutdown()
    except Exception as exc:
        print("⚠️ Could not measure VRAM using NVML:", exc)


thread = threading.Thread(target=monitor_vram, daemon=True)
thread.start()
print(f"🎨 {MODEL_NAME}: {WIDTH}x{HEIGHT}, {STEPS} steps, CFG {CFG_SCALE}, quant {QUANT_LABEL}")
print("Binary:", SD_BIN)
start = time.monotonic()
return_code = None
try:
    with log_path.open("w", encoding="utf-8") as logfile:
        process = subprocess.Popen(
            cmd, env=EXEC_ENV, cwd=str(BUILD_DIR),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors="replace", bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="")
                logfile.write(line)
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        return_code = process.wait()
finally:
    monitor_stop.set()
    thread.join(timeout=3)
elapsed = time.monotonic() - start

stats = {
    "model": MODEL_NAME,
    "model_variant": MODEL_VARIANT_NAME,
    "backend": "stable-diffusion.cpp (prebuilt CUDA binary from Drive)",
    "dit_file": MODEL_PATHS["dit"].name,
    "encoder_file": MODEL_PATHS["encoder"].name,
    "vae_file": MODEL_PATHS["vae"].name,
    "width": WIDTH, "height": HEIGHT, "seed": SEED,
    "steps": STEPS, "cfg_scale": CFG_SCALE,
    "sampler": "euler", "rng": "cpu",
    "flash_attention": USE_FLASH_ATTENTION, "vae_tiling": USE_VAE_TILING,
    "max_vram_gb": MAX_VRAM_GB,
    "generation_seconds": round(elapsed, 2),
    "peak_gpu_memory_mib": round(max(gpu_samples), 1) if gpu_samples else None,
    "exit_code": return_code,
    "prompt": PROMPT,
    "image": str(image_path),
    "log": str(log_path),
}
metadata_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"⏱️ Total time: {elapsed:.1f} seconds")
if gpu_samples:
    print(f"📈 GPU memory peak (NVML): {max(gpu_samples):.0f} MiB")
if return_code != 0 or not image_path.is_file():
    raise RuntimeError(
        f"sd-cli failed (exit code {return_code}). Check log: {log_path}. "
        "If OOM: reduce dimensions, select a lower quantization, or reduce MAX_VRAM_GB to 10. "
        "If unsupported model error: prebuilt binary on Drive is too old."
    )

if SAVE_IMAGES_TO_DRIVE:
    target_dir = pathlib.Path(DRIVE_RESULTS_DIR) / MODEL_SLUG
    target_dir.mkdir(parents=True, exist_ok=True)
    for p in (image_path, metadata_path, log_path):
        shutil.copy2(p, target_dir / p.name)
    print("💾 Results saved to Drive:", target_dir)

print("✅ Output image:", image_path)
print("📊 Benchmark JSON:", metadata_path)
display(Image.open(image_path))